# Dask and Xarray Best Practices

- Workshop: **Interactive (geospatial) data analysis at HPC scale with RS-DAT and Python**, NAC 2026 (Noordwijk)

- Date: April 9/10, 2026

## 1. Introduction

Dask and Xarray can facilitate interactive and explorative data analyses tasks at scale. However, they are not magic tools, and trying to "blindly" run parallel calculations on large datasets typically leads to poor performance or inefficient use of resources.

In this notebook, we try to showcase some of these tools' best practices, by providing an example were bad patterns have been intentionally been used.

We will work with a subset of the [Daymet](https://daac.ornl.gov/cgi-bin/dsviewer.pl?ds_id=1840) dataset, which includes daily weather data for North America (we will only focus on the Hawaii region). 

## 2. Monthly temperature analysis of Hawaii

In [ ]:
# You can connect to your Dask cluster here

Let's start by loading a few variables of the Daymet dataset over the Hawaii region. The data consists of a bunch of NetCDF/HDF5 files (one file per variable and year):

In [ ]:
import xarray as xr

In [ ]:
paths = "/project/remotesensing/Share/hpcdat_ws1/daymet-daily-v4/region-hi/daymet_v4_daily_hi_*_*.nc"

In [ ]:
ds = xr.open_mfdataset(
    paths,
    engine="h5netcdf",
    compat="override",
    coords="all",
    parallel=True,
    data_vars="all",
    drop_variables=("lat", "lon"),
    chunks={},
)

Let's consider only a subset of the full time extent (~40 years):

In [ ]:
ds = ds.sel(time=slice("2000-01", "2009-12"))

**Hint**: Inspect the dataset chunking scheme (`ds.chunks`)

In [ ]:
temperature = (ds["tmax"] + ds["tmin"]) / 2

**Hint**: the `temperature` variable will be reused multiple times in the notebook.

In [ ]:
t_mean_global = temperature.compute().mean(dim=("time", "x", "y"))
t_mean_global

**Hint**: Memory has just spiked when running the previous cell ... 

In [ ]:
t_mean = temperature.mean(dim="time").compute()

**Hint**: is the chunk shape appropriate?

In [ ]:
t_mean.plot.imshow()

In [ ]:
t_mean_monthly = temperature.resample(time="MS").mean()

In [ ]:
t_mean_monthly = t_mean_monthly.compute()
for time in t_mean_monthly.time:
    month = time.dt.strftime("%Y-%m").values
    subset = t_mean_monthly.sel(time=time)
    subset.to_netcdf(f"month_{month}.nc")

**Hint**: try using the Zarr format (see `.to_zarr()` method of Xarray objects).